<a href="https://colab.research.google.com/github/RENSHUZHE/HKU-DQMC/blob/main/DQMC_Theory_1_Hubbard_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import itertools

# DQMC Starting with the Hubbard Model

# Nomenclature
---
DQMC stands for determinant quantum Monte Carlo, and the early references are these:

Blankenbecler, Scalapino and Suger, Monte Carlo calculations of coupled boson-fermion systems. I, [Phys. Rev. D 24, 2278 (1981)](https://journals.aps.org/prd/abstract/10.1103/PhysRevD.24.2278),

Scalapino and Suger, Monte Carlo calculations of coupled boson-fermion systems. II, [Phys. Rev. B 24, 4295 (1981)](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.24.4295), and

Hirsch, Two-dimensional Hubbard model: Numerical simulation study,[Phys. Rev. B 31, 4403 (1985)](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.31.4403),

and there are also a few very good lecture notes, for example,

Assaad and Evertz, World Line and Determinantal Quantum Monte Carlo Methods for Strongly Correlated Electron Systems, [book chapter in Computational Many-Particle Physics, Lecture Notes in Physics, 739, 277-356 (2008)](https://pawn.physik.uni-wuerzburg.de/~assaad/Reprints/assaad_evertz.pdf).

But we here decided to break down knowledge barrier with square lattice and honeycomb lattice Hubbard model examples, those we have discussed in the [Hartree-Fock mean-field theories](https://quantummc.xyz/teaching/mean-field/) chapter of our lecture. We will use these example to explain the DQMC algorithm in step by step manner.

The following text and codes are prepared by Mr. [Shuzhe Ren](https://quantummc.xyz/renshuzhe/), Dr. [Gaopei Pan](https://scholar.google.com.hk/citations?user=vc75saMAAAAJ&hl=zh-CN) and [ZYM](https://quantummc.xyz/members/zi-yang-meng/).


## Statistical Mechanics
---
For classical models, we have the partition function:

$$Z=\sum_{\sigma} e^{-\beta E_{\sigma}}.$$

Physical observables:

$$\langle O\rangle=\frac{1}{Z} \sum_{C} O e^{-\beta E_{C}}.$$

For quantum systems, the partition function is:

$$Z=\operatorname{Tr}\left\{e^{-\beta H}\right\}.$$

Choosing a basis:

$$Z=\sum_{\{\phi\}}\left\langle\{\phi\}\left|e^{-\beta H}\right|\{\phi\}\right\rangle,$$

and physical observables become:

$$\langle O\rangle=\frac{\operatorname{Tr}\left\{O e^{-\beta H}\right\}}{Z}.$$

### Trotter Decomposition
---
Considering a system with a Hamiltonian $H = H_1 + H_2$, where $H_1$ could represent the free Hamiltonian and $H_2$ the interaction Hamiltonian. The partition function can be expressed as:

$$
\begin{aligned}
Z &=\operatorname{Tr}\left[e^{-\beta H}\right] \\
&=\operatorname{Tr}\left[(e^{-\Delta \tau H_1} e^{-\Delta \tau H_2})^{m} \right] \\
&=\operatorname{Tr}\left[e^{-\Delta \tau H_1} e^{-\Delta \tau H_2} \ldots e^{-\Delta \tau H_1} e^{-\Delta \tau H_2}\right] +\mathcal{O}\left[(\Delta \tau)^{2}\right]
\end{aligned}
\quad (1)
$$

where $m = \frac{\beta}{\Delta \tau}$ represents the number of imaginary time slices.

#### Error Analysis
In all numerical computations, tracking and minimising systematic errors is crucial. Because the operators $H_1$ and $H_2$ generally do not commute, splitting the exponential introduces a truncation error. A priori, the systematic error of the standard Trotter decomposition in Equation (1) is of order $\Delta\tau$. However, by applying time-dependent perturbation theory, we can show that under certain physical conditions, this leading-order error perfectly vanishes.

For a single time step $\Delta\tau$, the standard decomposition yields:
$$e^{-\Delta\tau(H_1+H_2)} = e^{-\Delta\tau H_1}e^{-\Delta\tau H_2} - \frac{\Delta\tau^2}{2}[H_1, H_2] + \mathcal{O}(\Delta\tau^3)$$

We can rewrite this to isolate the commutator term inside the exponent:
$$e^{-\Delta\tau(H - \frac{\Delta\tau}{2}[H_1, H_2])} = e^{-\Delta\tau H_1}e^{-\Delta\tau H_2} + \mathcal{O}(\Delta\tau^3)$$

Exponentiating both sides to the power of $m$ (where $m\Delta\tau = \beta$), the $\mathcal{O}(\Delta\tau^3)$ error accumulates $m$ times, promoting the overall error to $\mathcal{O}(\Delta\tau^2)$:
$$e^{-\beta(H - \frac{\Delta\tau}{2}[H_1, H_2])} = \left[e^{-\Delta\tau H_1}e^{-\Delta\tau H_2}\right]^m + \mathcal{O}(\Delta\tau^2)$$

To evaluate the left-hand side, we treat $h_1 = -\frac{\Delta\tau}{2}[H_1, H_2]$ as a small perturbation to the unperturbed Hamiltonian $H$. Applying time-dependent perturbation theory in imaginary time, we expand the exponential to linear order in $h_1$:
$$\left[e^{-\Delta\tau H_1}e^{-\Delta\tau H_2}\right]^m = e^{-\beta H} + \frac{\Delta\tau}{2} \underbrace{\int_0^\beta d\tau e^{-(\beta-\tau)H} [H_1, H_2] e^{-\tau H}}_{\equiv A} + \mathcal{O}(\Delta\tau^2)$$

When computing the expectation value of a physical observable $O = O^\dagger$, the numerator becomes:
$$\text{Tr}\left[(e^{-\Delta\tau H_1}e^{-\Delta\tau H_2})^m O\right] = \text{Tr}[e^{-\beta H} O] + \frac{\Delta\tau}{2}\text{Tr}[AO] + \mathcal{O}(\Delta\tau^2)$$

We can determine the properties of the integral operator $A$ by taking its Hermitian adjoint:
$$A^\dagger = \int_0^\beta d\tau \left( e^{-(\beta-\tau)H} [H_1, H_2] e^{-\tau H} \right)^\dagger = -\int_0^\beta d\tau e^{-\tau H} [H_1, H_2] e^{-(\beta-\tau)H}$$
By performing the substitution $\tau' = \beta - \tau$, we recover the original integral, revealing that $A^\dagger = -A$.

Because $A$ is an anti-Hermitian operator, it follows that $\text{Tr}[A] = -\text{Tr}[A]$ and therefore $\text{Tr}[AO] = -\text{Tr}[AO]$. If the operators $H_1$, $H_2$, and the observable $O$ can be simultaneously represented by real matrices in a chosen basis, the trace must evaluate to a purely real number. The only real number satisfying $x = -x$ is zero. Consequently, the leading $\mathcal{O}(\Delta\tau)$ term vanishes entirely, and the systematic error is naturally improved to $\mathcal{O}(\Delta\tau^2)$.

#### The Symmetric Trotter Decomposition
While the conditions above *(Perturbative assumption and Simultaneous Realnesss)* conveniently cancel the first-order error, the standard decomposition in Equation (1) remains structurally flawed because the operator $e^{-\Delta\tau H_1} e^{-\Delta\tau H_2}$ is not strictly Hermitian.

To guarantee stability and a baseline error of $\mathcal{O}(\Delta\tau^2)$ regardless of the specific properties of the operators, it is vastly preferable to use the symmetric decomposition:
$$e^{-\Delta\tau(H_1+H_2)} = e^{-\Delta\tau H_1/2} e^{-\Delta\tau H_2} e^{-\Delta\tau H_1/2} + \mathcal{O}(\Delta\tau^3)$$

This symmetric form is manifestly Hermitian and preserves positivity.

## Understanding the Determinant in DQMC
---

DQMC is primarily used to simulate finite-temperature fermionic systems. In practice, the algorithm only directly handles quadratic (two-fermion) terms. Therefore, for Hamiltonians containing quartic (four-fermion) interactions, such as the Hubbard model, we typically apply the Hubbard-Stratonovich transformation. This method introduces an auxiliary field to decouple the interactions, reducing the Hamiltonian to a non-interacting, two-fermion form coupled to a fluctuating background field. (As we will see later, the specific choice of decoupling method determines whether or not the system suffers from the sign problem).

<font color="red">**Determinantal or Detrimental depends your perspective ┐(´︎_`)┌ , :).**</font>

The method is called "Determinantal" Quantum Monte Carlo because it evaluates the many-body trace by converting it into the determinant of a matrix. For instance, suppose we want to calculate a trace of the following form:

$$\operatorname{Tr}\left[e^{-\sum_{i, j} \hat{c}_{i}^{\dagger} A_{i, j} \hat{c}_{j}}\right]=\operatorname{Det}\left[\mathbf{1}+e^{-\mathbf{A}}\right]. \tag{1}$$

Here, $A_{i, j}$ represents the elements of the single-particle matrix $\mathbf{A}$. We can see that the operation of calculating the trace over the many-body Fock space is mapped exactly to calculating the determinant of the identity matrix $\mathbf{1}$ plus the matrix $e^{-\mathbf{A}}$.

There are various ways to prove this identity. One approach is to diagonalise the operator in the exponent, noting that the trace is simply the sum over all possible particle occupations $n_{k} \in \{0,1\}$:

$$\begin{aligned}&\operatorname{Tr}\left[e^{-\sum_{i, j} \hat{c}_{i}^{\dagger} A_{i, j} \hat{c}_{j}}\right]\\ =&\operatorname{Tr}\left[e^{-\sum_{k} \epsilon(k) \hat{c}_{k}^{\dagger} \hat{c}_{k}}\right]\\=& \operatorname{Tr}\left[\Pi_{k} e^{-\epsilon(k) \hat{c}_{k}^{\dagger} \hat{c}_{k}}\right] \\ =&\Pi_{k} \operatorname{Tr}\left[e^{-\epsilon(k) \hat{c}_{k}^{\dagger} \hat{c}_{k}}\right]\\ =&\Pi_{k}\left[1+e^{-\epsilon(k)}\right]\end{aligned}$$

Since the determinant of a matrix is precisely the product of its eigenvalues, the equivalence holds naturally.

Alternatively, if manipulating operators in the exponent feels indirect, we can Taylor expand the exponential:

$$\exp \left(-c_{k}^{\dagger} \epsilon(k) c_{k}\right)=1-c_{k}^{\dagger} \epsilon(k) c_{k}+\frac{1}{2 !} c_{k}^{\dagger} \epsilon(k) c_{k} c_{k}^{\dagger} \epsilon(k) c_{k}-\frac{1}{3 !} \cdots$$

By applying the fermionic anticommutation relation $c_{k}^{\dagger} c_{k}+c_{k} c_{k}^{\dagger}=1$ alongside the nilpotent property $c_{k} c_{k} = 0$, we find:

$$\begin{aligned}
\exp \left(-c_{k}^{\dagger} \epsilon(k) c_{k}\right) &=1-c_{k}^{\dagger} \epsilon(k) c_{k}+\frac{1}{2 !} c_{k}^{\dagger} \epsilon(k)^{2} c_{k}-\frac{1}{3 !} \cdots \\
&=1+\left[e^{-\epsilon(k)}-1\right] c_{k}^{\dagger} c_{k}
\end{aligned}$$

(This reflects the well-known fermionic property that $\hat{n}_{k}^{m}=\hat{n}_{k}$ for any integer $m \geq 1$).

What is crucial for DQMC, however, is the generalised form of the above Eq.$~(1)$. For a product of exponentials, we can still perform a similar diagonalisation (as shown in Appendix A of *Hirsch, Two-dimensional Hubbard model: Numerical simulation study*, [Phys. Rev. B 31, 4403 (1985)](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.31.4403)):

$$e^{-\sum_{i, j} c_{i}^{\dagger} A_{i, j} c_{j}} e^{-\sum_{i, j} c_{i}^{\dagger} B_{i, j} c_{j}}=e^{-\sum_{\nu} c_{\nu}^{\dagger} l_{\nu} c_{\nu}}. \tag{2}$$

To prove Eq.$~(2)$, we will demonstrate that both the left-hand side (LHS) and right-hand side (RHS) produce the identical result when acting on any state within the occupation number basis of the Fock space.

Starting with a single-particle state $|\psi\rangle=c_{\nu}^{\dagger}|0\rangle$, the combined operator acts as follows:

$$e^{-\sum_{i, j} c_{i}^{\dagger} A_{i, j} c_{j}} e^{-\sum_{i, j} c_{i}^{\dagger} B_{i, j} c_{j}}|\psi\rangle=\left(e^{-A} e^{-B}\right)_{\nu \nu} c_{\nu}^{\dagger}|0\rangle=e^{-l_{\nu}} c_{\nu}^{\dagger}|0\rangle$$

Next, considering a many-particle state, such as the two-particle state $|\phi\rangle=c_{\mu_{1}}^{\dagger} c_{\mu_{2}}^{\dagger}|0\rangle$, we have:

$$\begin{aligned}
e^{-\sum_{i, j} c_{i}^{\dagger} B_{i j} c_{j}}|\phi\rangle &=\prod_{\mu}\left[1+\left(e^{-B_{\mu}}-1\right) c_{\mu}^{\dagger} c_{\mu}\right] c_{\mu_{1}}^{\dagger} c_{\mu_{2}}^{\dagger}|0\rangle \\
&=e^{-B_{\mu_{1}}} e^{-B_{\mu_{2}}} c_{\mu_{1}}^{\dagger} c_{\mu_{2}}^{\dagger}|0\rangle
\end{aligned}$$

The same logic applies to the operator involving $\mathbf{A}$. Hence, the two-particle basis state also satisfies Eq.$~(2)$.

Lastly, by applying this argument repeatedly, we can generalise this to an arbitrary $N$-particle state $|\Phi\rangle = c_{\mu_{1}}^{\dagger} c_{\mu_{2}}^{\dagger} \dots c_{\mu_{N}}^{\dagger} |0\rangle$:

$$e^{-\sum c^{\dagger} A c} e^{-\sum c^{\dagger} B c} |\Phi\rangle = e^{-l_{\mu_{1}}} e^{-l_{\mu_{2}}} \dots e^{-l_{\mu_{N}}} |\Phi\rangle$$

Therefore, by invoking the trace-determinant relationship Eq.$~(1)$ established earlier, we arrive at the central identity:

$$\operatorname{Tr}\left[e^{-\sum_{i, j} c_{i}^{\dagger} A_{i, j} c_{j}} e^{-\sum_{i, j} c_{i}^{\dagger} B_{i, j} c_{j}}\right]=\operatorname{Det}\left(\mathbf{1}+e^{-\mathbf{A}} e^{-\mathbf{B}}\right).$$

For products involving more than two exponential operators, the generalised form follows naturally by repeatedly applying the same logic.

### Numerics verifying: Hubbard Model

The Hubbard model provides a fundamental description of interacting electrons on a lattice. The full Hamiltonian consists of two competing terms: the kinetic energy, which allows electrons to hop between nearest-neighbour sites, and the potential energy, which penalises two electrons of opposite spin occupying the same site. It is conventionally written as:

$$H = -t \sum_{\langle i,j \rangle, \sigma} \left( \hat{c}_{i\sigma}^{\dagger} \hat{c}_{j\sigma} + \hat{c}_{j\sigma}^{\dagger} \hat{c}_{i\sigma} \right) + U \sum_{i} \hat{n}_{i\uparrow} \hat{n}_{i\downarrow}$$

where $t$ is the hopping amplitude, $U$ is the on-site Coulomb repulsion, $\sigma \in \{\uparrow, \downarrow\}$ is the spin index, and $\langle i,j \rangle$ denotes summation over nearest-neighbour pairs. Again, I enforced the periodic boundary condition here.

In the special case where $U = 0$, the interaction term vanishes entirely. The Hamiltonian reduces to a purely quadratic, non-interacting tight-binding model. Because the electrons no longer interact with one another, they behave as a free Fermi gas on a discrete lattice. This non-interacting Hamiltonian is exactly solvable; by applying a Fourier transform to momentum space, the single-particle energy levels can be completely diagonalised, yielding the standard tight-binding band structure.

In [2]:
#parameters for the U=0, 4x4 square lattice
L = 4
N = L * L
t = 1.0
beta = 1.0

print(f"--- Verifying Trace-Determinant Identity for U=0, beta={beta}, L={L}x{L} ---")

# Initiate single-particle matrix h
h = np.zeros((N, N))

for x in range(L):
    for y in range(L):
        i = x * L + y
        # Kinetic Term: Nearest-neighbour hopping with periodic boundaries
        j_x = ((x + 1) % L) * L + y
        h[i, j_x] -= t
        h[j_x, i] -= t

        j_y = x * L + ((y + 1) % L)
        h[i, j_y] -= t
        h[j_y, i] -= t

# RHS (Determinant Formula)
# Diagonalise h
vals, vecs = np.linalg.eigh(h)

# Spectral thm
exp_minus_beta_h = vecs @ np.diag(np.exp(-beta * vals)) @ vecs.T

rhs_det = np.linalg.det(np.eye(N) + exp_minus_beta_h)
print(f"RHS (Determinant of matrix): {rhs_det:.8e}")

# LHS
lhs_trace = 0.0

# Generate all combinations for the Fock space occupations
for occ in itertools.product([0, 1], repeat=N):
    # The energy of the many-body state is the sum of occupied single-particle energies
    E_manybody = np.dot(occ, vals)
    lhs_trace += np.exp(-beta * E_manybody)

print(f"LHS (Explicit Trace over {2**N} states): {lhs_trace:.8e}")

# Step 4: Compare the results
diff = np.abs(rhs_det - lhs_trace)
print(f"Absolute Difference: {diff:.8e}")

--- Verifying Trace-Determinant Identity for U=0, beta=1.0, L=4x4 ---
RHS (Determinant of matrix): 2.98175298e+07
LHS (Explicit Trace over 65536 states): 2.98175298e+07
Absolute Difference: 2.09733844e-06



## Hubbard Model and Hubbard-Stratonovitch Transformation

The Matsubara Green's function for free electrons reads:

$$G\left(i \omega_{n}\right)=\frac{1}{i \omega_{n}-\left(\epsilon_{k}-\mu\right)}.$$

However, for models containing four-fermion interactions like the Hubbard Model, we need a discrete Hubbard-Stratonovitch decomposition, introducing an auxiliary field to carry out Monte Carlo solving.

Historically, this type of DQMC was first used to solve boson-fermion coupling models, also known as the BSS algorithm, see  Blankenbecler, Scalapino and Suger, Monte Carlo calculations of coupled boson-fermion systems. I, [Phys. Rev. D 24, 2278 (1981)](https://journals.aps.org/prd/abstract/10.1103/PhysRevD.24.2278). Later, in 1983, Hirsch proposed the HS transformation for the Hubbard model with on-site Coulomb interactions, that is, by introducing an auxiliary field, decoupling the interaction into the coupling of non-interacting fermions and the auxiliary field, which gradually enabled simple DQMC calculations for interacting fermion systems.

The Hamiltonian of the Hubbard Model is written as:

$$\hat{H}=-t \sum_{\langle i j\rangle \sigma} \hat{c}_{i \sigma}^{\dagger} \hat{c}_{j \sigma}+h.c.+U \sum_{i}\left(\hat{n}_{i \uparrow}-\frac{1}{2}\right)\left(\hat{n}_{i \downarrow}-\frac{1}{2}\right).$$

We usually divide the imaginary time into $M$ slices: $\beta=M \Delta \tau$. Then we perform a Trotter decomposition. The reason for doing this is because $\hat{H}_{0}=-t \sum_{\langle i j\rangle \sigma} \hat{c}_{i \sigma}^{\dagger} \hat{c}_{j \sigma}+h.c.$ and $\hat{H}_{I}=U \sum_{i}\left(\hat{n}_{i \uparrow}-\frac{1}{2}\right)\left(\hat{n}_{i \downarrow}-\frac{1}{2}\right)$ do not commute.

We want to perform the HS transformation on $e^{-\Delta \tau \hat{H}_{I}}$, so we need to separate it out:

$$e^{-\Delta \tau \hat{H}}=e^{-\Delta \tau\left(\hat{H}_{0}+\hat{H}_{I}\right)}=e^{-\Delta \tau \hat{H}_{0}} e^{-\Delta \tau \hat{H}_{I}}+\mathcal{O}\left[(\Delta \tau)^{2}\right].$$

As for why the error is $\mathcal{O}\left[(\Delta \tau)^{2}\right]$, there are multiple ways to understand it,  <font color="red"> (Shuzhe, here you can also discuss the trotter error estimation here.)</font>

So what exactly is the HS transformation? In a sense, it is an inverse Gaussian integral.

In our freshman thermodynamics class, we should have learned the formula for the Gaussian integral:

$$e^{A^{2} / 2}=\frac{1}{\sqrt{2 \pi}} \int_{-\infty}^{+\infty} \mathrm{d}\phi e^{-\frac{\phi^{2}}{2}-\phi A}.$$

And at this point we know that the Hubbard U term can be written in a squared form:

$$H_{U}=-\frac{U}{2}\left(n_{\uparrow}-n_{\downarrow}\right)^{2}+\frac{U}{4}$$

Then, we simply substitute $A^{2}=\Delta \tau U\left(n_{\uparrow}-n_{\downarrow}\right)^{2}$ into the previous Gaussian integral formula.

In this way, we find that by introducing $\phi$, we have changed the form of $A^{2}$ into the form of $A \phi$, which is a linear term of $A$.

Interestingly, for the Hubbard model, we have a more clever way to decouple:

$$e^{-\Delta \tau H_{U}}=\gamma \sum_{s=\pm 1} e^{\alpha s\left(n_{\uparrow}-n_{\downarrow}\right)},$$

where $\gamma=\frac{1}{2} e^{-\Delta \tau U / 4}$ and $\cosh(\alpha)=e^{\Delta \tau U / 2}$.

The coefficients can be obtained by considering $e^{-\Delta \tau H_{U}}$ in the Hilbert space of a single lattice site:

$$\begin{aligned}
e^{-\Delta \tau U / 4}|0\rangle &=2\gamma|0\rangle \\
e^{-\Delta \tau U / 4}|\uparrow\downarrow\rangle &=2\gamma|\uparrow\downarrow\rangle \\
e^{\Delta \tau U / 4}|\uparrow\rangle &=2\gamma\cosh(\alpha)|\uparrow\rangle \\
e^{\Delta \tau U / 4}|\downarrow\rangle &=2\gamma\cosh(\alpha)|\downarrow\rangle.
\end{aligned}$$

This method breaks the $SU(2)$ symmetry of the spin, and we could alternatively choose:

$$e^{-\Delta \tau H_{U}}=\tilde{\gamma} \sum_{s=\pm 1} e^{i\tilde{\alpha}s\left(n_{\uparrow}+n_{\downarrow}-1\right)},$$

where $\cos(\tilde{\alpha})=e^{-\Delta \tau U/2}$ and $\tilde{\gamma}=\frac{1}{2}e^{\Delta \tau U/4}$. The cost is the introduction of complex numbers.

The situation of the Hubbard Model here is quite special, which is why it can be decoupled by introducing an auxiliary field of $\pm 1$. For more general four-fermion interactions (requiring hermiticity), we have more general HS transformation methods, such as introducing auxiliary fields of $\pm 1$, $\pm 2$:

$$e^{\Delta \tau\lambda A^{2}}=\sum_{l=\pm 1,\pm 2} \frac{\gamma(l)}{4} e^{\sqrt{\Delta \tau\lambda}\eta(l)O} +\mathcal{O}\left(\Delta\tau^{4}\right),$$

where

$$\begin{aligned}
\gamma(\pm 1)=1+\sqrt{6}/3, & \qquad \gamma(\pm 2)=1-\sqrt{6}/3 \\
\eta(\pm 1)=\pm\sqrt{2(3-\sqrt{6})}, &\qquad \eta(\pm 2)=\pm\sqrt{2(3+\sqrt{6})}
\end{aligned}.$$

This method is not exact and contains an error related to $\Delta\tau$. For observables, the resulting error is proportional to $\Delta\tau^{3}$.  <font color="red"> (Shuzhe, you can explain why the trotter error estimation gives cubic term here.)</font>

And how are these current coefficients solved for? I feel that everywhere they just directly give the results, and rarely does anyone manually explain where they come from. Here we might as well assume:

$$\gamma(1)=\gamma(-1)=a, \quad \gamma(2)=\gamma(-2)=b, \quad \eta(1)=\sqrt{c}=-\eta(-1), \quad \eta(2)=\sqrt{d}=-\eta(-2)$$

Then we Taylor expand both sides up to $\mathcal{O}\left(p^{4}\right)$, giving:

$$\begin{array}{c}
1=\frac{1}{2}(a+b) \\
1=\frac{1}{4}(ac+bd) \\
\frac{1}{2}=\frac{1}{48}\left(ac^{2}+bd^{2}\right) \\
\frac{1}{6}=\frac{1}{1440}\left(ac^{3}+bd^{3}\right)
\end{array}$$

Solving this yields:

$$\begin{array}{ll}
a=1+\sqrt{6}/3, & b=1-\sqrt{6}/3 \\
c=2(3-\sqrt{6}), & d=2(3+\sqrt{6})
\end{array}$$

We will know later that different decoupling methods may lead to the presence or absence of the sign problem. The so-called sign problem here means that for a certain configuration of the auxiliary field, the corresponding weight is a complex or negative number. There is a good reference: the sign problem in quantum Monte Carlo simulations, Gaopei Pan, Zi Yang Meng,
[Book Volume 1 for Encyclopedia of Condensed Matter Physics, Pages 867-878 (2023)](https://www.sciencedirect.com/science/chapter/referencework/abs/pii/B9780323908009000950).

The next section will derive how to calculate the corresponding Weight for a specific configuration of the auxiliary field after introducing the auxiliary field.



## The Weights of the Hubbard Model

We already know from the previous text that when $H=H_{0}+H_{I}$ and usually $[H_{0}, H_{I}] \neq 0$, we similarly divide the imaginary time into $L_{\tau}$ slices: $\beta=L_{\tau}\Delta\tau$, and perform the Trotter decomposition:

$$\begin{aligned}
Z &=\operatorname{Tr}\left[e^{-\beta\hat{H}}\right] \\
&=\operatorname{Tr}\left[\left(e^{-\Delta\tau\hat{H}}\right)^{L_{\tau}}\right] \quad \text { where } e^{-\Delta\tau\hat{H}} \approx e^{-\Delta\tau\hat{H}_{I}} e^{-\Delta\tau\hat{H}_{0}}
\end{aligned}$$

Namely:

$$Z=\operatorname{Tr}\left[e^{-\Delta\tau\hat{H}_{I}} e^{-\Delta\tau\hat{H}_{0}} e^{-\Delta\tau\hat{H}_{I}} e^{-\Delta\tau\hat{H}_{0}} \cdots e^{-\Delta\tau\hat{H}_{I}} e^{-\Delta\tau\hat{H}_{0}}\right]$$

And from the previous text we already know, for the Hubbard U term, we have:

$$\begin{aligned}
e^{-\Delta\tau\hat{H}_{I}} &=\prod_{i} e^{-\Delta\tau U\left(\hat{n}_{i\uparrow}-\frac{1}{2}\right)\left(\hat{n}_{i\downarrow}-\frac{1}{2}\right)} \\
&=\prod_{i} \gamma \sum_{s_{i,l}=\pm 1} e^{\alpha s_{i,l}\left(\hat{n}_{i\uparrow}-\hat{n}_{i\downarrow}\right)} \\
&=\gamma^{N} \sum_{s_{i,l}=\pm 1}\left(\prod_{i} e^{\alpha s_{i,l}\hat{n}_{i\uparrow}} \prod_{i} e^{-\alpha s_{i,l}\hat{n}_{i\downarrow}}\right)
\end{aligned}$$

Where $\gamma=\frac{1}{2}e^{-\Delta\tau U/4}, \quad \cosh(\alpha)=e^{\Delta\tau U/2}$. $l$ is the label in the imaginary time direction. Here $N$ is the number of lattice sites $L \times L$.

For the sake of notational simplicity, we have the following notation: $\hat{H}_{0}=\boldsymbol{c}^{\dagger} T\boldsymbol{c}=-t \sum_{\langle ij\rangle\sigma} \hat{c}_{i\sigma}^{\dagger}\hat{c}_{j\sigma}+h.c.$, where $T$ is a matrix that only has matrix elements of $-t$ at nearest neighbours. In practice, a more convenient way to write this is to only write out the spin up or down part. The symbol $T$ hereafter in this text will only represent the partitioned small matrix within the large matrix, which is:

$$\boldsymbol{c}_{\uparrow}^{\dagger} T\boldsymbol{c}_{\uparrow}=-t \sum_{\langle ij\rangle} \hat{c}_{i\uparrow}^{\dagger}\hat{c}_{j\uparrow}+h.c. \quad \boldsymbol{c}_{\downarrow}^{\dagger} T\boldsymbol{c}_{\downarrow}=-t \sum_{\langle ij\rangle} \hat{c}_{i\downarrow}^{\dagger}\hat{c}_{j\downarrow}+h.c.$$

At this point, the partition function is written as:

$$Z =\sum_{s_{i,l}=\pm 1} \gamma^{NL_{\tau}} \operatorname{Tr}_{F}\left\{\prod_{l=1}^{L_{\tau}}\left[\left(\prod_{i} e^{\alpha s_{i,l}\hat{n}_{i\uparrow}}\right)\left(e^{-\Delta\tau c_{\uparrow}^{\dagger}Tc_{\uparrow}}\right)\left(\prod_{i} e^{-\alpha s_{i,l}\hat{n}_{i\downarrow}}\right)\left(e^{-\Delta\tau c_{\downarrow}^{\dagger}Tc_{\downarrow}}\right)\right]\right\}$$

It can clearly be seen that the form on the exponent of e is partitioned with respect to spin up and down, so it can be written in a more convenient form, and according to:

$$\operatorname{Tr}\left[e^{-\sum_{i,j} c_{i}^{\dagger}A_{i,j}c_{j}} e^{-\sum_{i,j} c_{i}^{\dagger}B_{i,j}c_{j}}\right]=\operatorname{Det}\left(1+e^{-\mathbf{A}}e^{-\mathbf{B}}\right)$$

we know that the result obtained by calculating the Trace over the fermion part is equivalent to the determinant of the exp of the corresponding matrix. At this time we have:

$$Z=\gamma^{NL_{\tau}} \sum_{\{s_{i,l}\}} \prod_{\sigma=\uparrow\downarrow} \operatorname{Det}\left[\mathbf{I}+\mathbf{B}^{\sigma}\left(L_{\tau}\Delta\tau,(L_{\tau}-1)\Delta\tau\right) \cdots \mathbf{B}^{\sigma}(\Delta\tau,0)\right]$$

Where:

$$\begin{array}{l}
\mathbf{B}^{\uparrow}\left(l_{2}\Delta\tau, l_{1}\Delta\tau\right)=\prod_{l=l_{1}+1}^{l_{2}} e^{\alpha\operatorname{Diag}(\vec{S}_{l})} e^{-\Delta\tau T} \\
\mathbf{B}^{\downarrow}\left(l_{2}\Delta\tau, l_{1}\Delta\tau\right)=\prod_{l=l_{1}+1}^{l_{2}} e^{-\alpha\operatorname{Diag}(\vec{S}_{l})} e^{-\Delta\tau T}
\end{array}$$

Where $\operatorname{Diag}(\vec{S}_{l})$ represents a matrix whose diagonal elements are respectively $s_{i,l}$.

Now we see that we have traced out the fermions, and the summation part of the partition function is merely a sum over an $L \times L \times L_{\tau}$ Ising auxiliary field. We temporarily set aside the matter of the auxiliary field; just like the Ising model, in the Ising model, when we are given a certain configuration of the Ising field, we obtain the weight (probability) corresponding to that configuration. Here, similarly, given the values of the $L \times L \times L_{\tau}$ Ising field, we have the corresponding weight:

$$W_{C}=\frac{\gamma^{NL_{\tau}} \prod_{\sigma=\uparrow\downarrow} \operatorname{Det}\left[\mathbf{I}+\mathbf{B}_{C}^{\sigma}\left(L_{\tau}\Delta\tau,(L_{\tau}-1)\Delta\tau\right) \cdots \mathbf{B}_{C}^{\sigma}(\Delta\tau,0)\right]}{Z}$$

Where adding $C$ refers to substituting the values of a specific Configuration to obtain the corresponding matrix before performing operations. For example, looking at the part $\mathbf{B}_{C}^{\uparrow}(\Delta\tau,0)=e^{\alpha\operatorname{Diag}(\vec{S}_{1})}e^{-\Delta\tau T}$, we find that the latter half is independent of the Ising field, while the former half depends on whether the value of $s_{i,1}$ is $+1$ or $-1$.

We know that when performing MCMC, what truly affects the computation is the ratio of weights, so we can omit identical constant factors and rewrite the weight as:

$$W_{C}=\prod_{\sigma=\uparrow\downarrow} \operatorname{Det}\left[\mathbf{I}+\mathbf{B}_{C}^{\sigma}\left(L_{\tau}\Delta\tau,(L_{\tau}-1)\Delta\tau\right) \cdots \mathbf{B}_{C}^{\sigma}(\Delta\tau,0)\right]=\prod_{\sigma=\uparrow\downarrow} \operatorname{Det}\left[\mathbf{I}+ \mathbf{B}_{C}^{\sigma}(\beta,0)\right]$$

Now we see that, given a specific auxiliary field: a configuration of the 2+1 Ising field, we can calculate the weight of that configuration (although it is very troublesome). We need to calculate each $\mathbf{B}$, which requires calculating the exp of a large matrix. The calculation of the exp of a matrix is usually done through diagonalisation, using what we learned in linear algebra: $e^{-A}=e^{-U^{\dagger}\operatorname{diag}(\lambda)U}=U^{\dagger}\operatorname{Diag}(e^{-\lambda_{i}})U$.

At this point, it is already a very strenuous operation, yet we still have to multiply the obtained $2L_{\tau}$ matrices together (which may lead to numerical errors during the multiplication process). After multiplying them, we must add an identity matrix, and then calculate the determinant of this massive $N \times N$ matrix. This is yet another strenuous operation. And after performing all these operations, we have now only calculated a single number: the weight corresponding to that configuration. And when you attempt to update to a new configuration and want to calculate the weight of the new configuration, thinking trivially, you would need to recalculate each $\mathbf{B}$ separately, multiply them together, and then calculate the determinant... This is obviously a terrifying amount of computation. Furthermore, directly multiplying $2L_{\tau}$ matrices that may have very large singularities will obviously easily cause numerical errors.

These problems will be solved one by one later, but at least now you can carry out the DQMC calculation of the Hubbard Model in an incredibly brute-force manner.